# EDA: Analisis Input Features untuk Sistem Prediksi

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

try:
    import seaborn as sns
    HAVE_SEABORN = True
except ModuleNotFoundError:
    HAVE_SEABORN = False
    print('seaborn tidak terpasang.')

DATASET_DIR = Path('dataset')
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COL_YIELD = 'hg/ha_yield'
COL_RAINFALL = 'average_rain_fall_mm_per_year'
COL_PEST = 'pesticides_tonnes'
COL_TEMP = 'avg_temp'
COL_ITEM = 'Item'
COL_AREA = 'Area'
COL_YEAR = 'Year'

preferred_files = ['yield_df.csv', 'crop_production.csv', 'yield.csv']
csv_path = None
for name in preferred_files:
    candidate = DATASET_DIR / name
    if candidate.exists():
        csv_path = candidate
        break

if csv_path is None:
    all_csv = sorted(DATASET_DIR.glob('*.csv'))
    csv_path = all_csv[0]

print('Membaca dataset:', csv_path)
df = pd.read_csv(csv_path)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print('Shape:', df.shape)

seaborn tidak terpasang.
Membaca dataset: dataset\yield_df.csv
Shape: (28242, 7)

In [2]:
BLUE = '#2563EB'
TEAL = '#0891B2'
PURPLE = '#7C3AED'
AMBER = '#D97706'
SLATE = '#475569'
LIGHT = '#E2E8F0'
GRID = '#CBD5E1'
RED = '#EF4444'
GREEN = '#10B981'
ORANGE = '#F97316'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': GRID,
    'grid.linewidth': 0.6,
    'axes.labelsize': 11,
    'figure.dpi': 150,
})

num_cols = [COL_YIELD, COL_RAINFALL, COL_PEST, COL_TEMP]
col_labels = {
    COL_YIELD: 'Yield (hg/ha)',
    COL_RAINFALL: 'Curah Hujan (mm/tahun)',
    COL_PEST: 'Pestisida (ton)',
    COL_TEMP: 'Suhu (degC)',
    COL_ITEM: 'Jenis Tanaman',
    COL_AREA: 'Negara',
}

## BAGIAN 1: ANALISIS OUTLIER

In [3]:
def detect_outliers_iqr(data, multiplier=1.5):
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    return (data < lower) | (data > upper), lower, upper

def detect_outliers_zscore(data, threshold=3):
    z = np.abs(stats.zscore(data))
    return z > threshold

print('=' * 60)
print('ANALISIS OUTLIER')
print('=' * 60)

outlier_summary = {}
for col in [COL_YIELD, COL_RAINFALL, COL_PEST, COL_TEMP]:
    data = df[col].dropna()
    outlier_iqr, lower_iqr, upper_iqr = detect_outliers_iqr(data)
    outlier_z = detect_outliers_zscore(data)
    outlier_summary[col] = {
        'count_iqr': outlier_iqr.sum(),
        'pct_iqr': outlier_iqr.sum() / len(data) * 100,
    }
    print(f'{col_labels[col]}:')
    print(f'  Range: {data.min():.2f} - {data.max():.2f}')
    print(f'  IQR bounds: [{lower_iqr:.2f}, {upper_iqr:.2f}]')
    print(f'  Outliers (IQR): {outlier_iqr.sum()} ({outlier_iqr.sum()/len(data)*100:.2f}%)')

In [4]:
print('\n' + '=' * 60)
print('OUTLIER PER JENIS TANAMAN')
print('=' * 60)

outlier_per_item = {}
for item in sorted(df[COL_ITEM].unique()):
    item_data = df[df[COL_ITEM] == item][COL_YIELD].dropna()
    outlier_iqr, _, _ = detect_outliers_iqr(item_data)
    outlier_per_item[item] = {
        'count': outlier_iqr.sum(),
        'pct': outlier_iqr.sum() / len(item_data) * 100,
        'total': len(item_data),
    }
    print(f'{item}: {outlier_iqr.sum()}/{len(item_data)} outliers ({outlier_iqr.sum()/len(item_data)*100:.1f}%)')

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, col in enumerate([COL_YIELD, COL_RAINFALL, COL_PEST, COL_TEMP]):
    ax = axes.flatten()[i]
    data = df[col].dropna()
    outlier_iqr, lower, upper = detect_outliers_iqr(data)
    bp = ax.boxplot(data, vert=True, patch_artist=True)
    bp['boxes'][0].set_facecolor(LIGHT)
    bp['boxes'][0].set_edgecolor(BLUE)
    bp['medians'][0].set_color(RED)
    ax.axhline(upper, color=ORANGE, linestyle='--', linewidth=1)
    ax.set_title(f'{col_labels[col]}\nOutliers: {outlier_iqr.sum()} ({outlier_iqr.sum()/len(data)*100:.1f}%)', fontweight='bold')

plt.suptitle('Analisis Outlier', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gambar1_outlier_boxplot.png', bbox_inches='tight')
plt.show()

In [6]:
extreme_yield = df[df[COL_YIELD] > 400000][[COL_ITEM, COL_AREA, COL_YEAR, COL_YIELD]].sort_values(COL_YIELD, ascending=False)
print('OUTLIER EXTREME (Yield > 400,000 hg/ha):')
print(f'Jumlah: {len(extreme_yield)} records')
print(extreme_yield.to_string())

print('\nREKOMENDASI:')
print('1. Outlier Yield (Kentang): Jangan dihapus, gunakan log-transform')
print('2. Outlier Pesticides: Log-transform atau standardization')
print('3. Outlier Suhu/Hujan: Sangat kecil, bisa di-cap jika tidak valid')

## BAGIAN 2: VALIDASI INPUT USER

In [7]:
print('=' * 60)
print('ANALISIS INPUT UNTUK SISTEM PREDIKSI')
print('=' * 60)

print('\n[A] KOLOM KATEGORIKAL:')
print(f'  - Area: {df[COL_AREA].nunique()} unique negara')
print(f'  - Item: {df[COL_ITEM].nunique()} unique tanaman')

print('\n[B] KOLOM NUMERIK (RANGE DATA):')
for col in [COL_RAINFALL, COL_TEMP, COL_PEST]:
    print(f'  - {col_labels[col]}: Min={df[col].min():.1f}, Max={df[col].max():.1f}, Mean={df[col].mean():.1f}')

In [8]:
print('\n' + '=' * 60)
print('VALIDASI RANGE INPUT YANG WAJAR')
print('=' * 60)

valid_ranges = {
    COL_RAINFALL: {'min': 0, 'max': 3500},
    COL_TEMP: {'min': -10, 'max': 45},
    COL_PEST: {'min': 0, 'max': 500000},
}

print('\nRANGE YANG DISARANKAN UNTUK INPUT USER:')
for col, info in valid_ranges.items():
    data_range = df[col]
    outside = ((data_range < info['min']) | (data_range > info['max'])).sum()
    print(f'  {col_labels[col]}: {info["min"]} - {info["max"]} (data di luar: {outside})')

print('\nSARAN UNTUK APLIKASI:')
print('  - Validasi input di level aplikasi')
print('  - Tampilkan pesan error jika di luar range')

## BAGIAN 3: STRATEGI ENCODING

In [9]:
print('=' * 60)
print('STRATEGI ENCODING')
print('=' * 60)

cat_cols = [COL_AREA, COL_ITEM]
for col in cat_cols:
    n_unique = df[col].nunique()
    value_counts = df[col].value_counts()
    print(f'\n{col_labels[col]}: {n_unique} unique')
    print(f'  Min freq: {value_counts.min()}, Max freq: {value_counts.max()}')

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

area_counts = df[COL_AREA].value_counts()
axes[0].barh(area_counts.index[:15], area_counts.values[:15], color=BLUE)
axes[0].set_xlabel('Jumlah Records')
axes[0].set_title(f'Distribusi Negara (Top 15)', fontweight='bold')
axes[0].invert_yaxis()

item_counts = df[COL_ITEM].value_counts()
axes[1].bar(item_counts.index, item_counts.values, color=TEAL)
axes[1].set_xlabel('Jenis Tanaman')
axes[1].set_ylabel('Jumlah Records')
axes[1].set_title('Distribusi Tanaman', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gambar2_encoding_distribution.png', bbox_inches='tight')
plt.show()

In [11]:
print('\nSTRATEGI ENCODING YANG DISARANKAN:')
print('\n[A] AREA (101 unique):')
print('  - Cardinality: TINGGI')
print('  - SOLUSI: Target Encoding (mean yield per negara)')
print('  - Jika ada negara baru: gunakan global mean')

print('[B] ITEM (10 unique):')
print('  - Cardinality: RENDAH')
print('  - SOLUSI: One-Hot Encoding')

area_means = df.groupby(COL_AREA)[COL_YIELD].mean().sort_values(ascending=False)
print('\nContoh Target Encoding - Area (Top 5):')
for area in area_means.head(5).index:
    print(f'  {area}: {area_means[area]:,.0f}')

## BAGIAN 4: TRAIN-TEST SPLIT

In [12]:
print('=' * 60)
print('ANALISIS TRAIN-TEST SPLIT')
print('=' * 60)

years = sorted(df[COL_YEAR].unique())
print(f'\nRentang tahun: {min(years)} - {max(years)} ({len(years)} tahun)')

In [13]:
print('\nSTRATEGI SPLIT:')
print('\n[1] RANDOM SPLIT:')
print('  - PRO: Menggunakan semua data')
print('  - CONS: Data leakage temporal')

print('\n[2] TIME-BASED SPLIT (REKOMENDASI):')
print('  - Train: tahun lama, Test: tahun terbaru')
print('  - PRO: Tidak ada data leakage')
print('  - CONS: Model tidak lihat tren terbaru')

split_year = 2010
df_train = df[df[COL_YEAR] <= split_year]
df_test = df[df[COL_YEAR] > split_year]

print(f'\nContoh Split (Train <= {split_year}, Test > {split_year}):')
print(f'  Train: {len(df_train):,} records')
print(f'  Test: {len(df_test):,} records')

In [14]:
fig, ax = plt.subplots(figsize=(10, 5))

yearly_mean = df.groupby(COL_YEAR)[COL_YIELD].mean()
ax.plot(yearly_mean.index, yearly_mean.values, color=BLUE, linewidth=2, marker='o')
ax.fill_between(yearly_mean.index, yearly_mean.values, alpha=0.2, color=BLUE)
ax.axvline(split_year, color=RED, linestyle='--', linewidth=2)
ax.axvspan(1990, split_year, alpha=0.1, color=GREEN)
ax.axvspan(split_year, 2013, alpha=0.1, color=ORANGE)
ax.set_xlabel('Tahun')
ax.set_ylabel('Rata-rata Yield')
ax.set_title(f'Time-Based Split (Train: 1990-{split_year}, Test: {split_year+1}-2013)', fontweight='bold')
ax.text(2000, ax.get_ylim()[1]*0.95, 'TRAIN', ha='center', fontsize=10, fontweight='bold')
ax.text(2012, ax.get_ylim()[1]*0.95, 'TEST', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gambar3_traintest_split.png', bbox_inches='tight')
plt.show()

## BAGIAN 5: ANALISIS FITUR YEAR

In [15]:
print('=' * 60)
print('ANALISIS FITUR YEAR')
print('=' * 60)

correlation = df[COL_YEAR].corr(df[COL_YIELD])
spearman = df[COL_YEAR].corr(df[COL_YIELD], method='spearman')
print(f'\nKorelasi Year vs Yield:')
print(f'  Pearson: r = {correlation:.4f}')
print(f'  Spearman: rho = {spearman:.4f}')

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

yearly_mean = df.groupby(COL_YEAR)[COL_YIELD].mean()
axes[0].plot(yearly_mean.index, yearly_mean.values, color=BLUE, linewidth=2, marker='o')
axes[0].set_xlabel('Tahun')
axes[0].set_ylabel('Rata-rata Yield')
axes[0].set_title(f'Trend Yield (r={correlation:.3f})', fontweight='bold')

item_trends = df.groupby([COL_YEAR, COL_ITEM])[COL_YIELD].mean().unstack()
for item in item_trends.columns:
    axes[1].plot(item_trends.index, item_trends[item], marker='o', markersize=3, label=item)
axes[1].set_xlabel('Tahun')
axes[1].set_ylabel('Rata-rata Yield')
axes[1].set_title('Trend per Jenis Tanaman', fontweight='bold')
axes[1].legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gambar4_year_trends.png', bbox_inches='tight')
plt.show()

In [17]:
print('\nREKOMENDASI PENGGUNAAN FITUR YEAR:')
print('\n[OPSI A] TIDAK MASUKKAN YEAR')
print('  - Alasan: User tidak tahu tahun depan')
print('  - Gunakan kondisi lingkungan saja')

print('\n[OPSI B] MASUKKAN YEAR')
print('  - Alasan: Ada tren temporal')
print('  - Masalah: Tidak bisa prediksi tahun > 2013')

print('\n[KESIMPULAN]: JANGAN gunakan Year sebagai fitur input')
print('Gunakan fitur lingkungan (suhu, hujan, pestisida) saja')

## RINGKASAN

In [18]:
print('=' * 60)
print('RINGKASAN TEMUAN')
print('=' * 60)

print('\n1. OUTLIER:')
print('   - Yield: 2.7% outliers (terutama Kentang >400k)')
print('   - Rekomendasi: Log-transform, JANGAN dihapus')

print('\n2. INPUT VALIDATION:')
print('   - Area: 101 negara (dropdown)')
print('   - Item: 10 tanaman (dropdown)')
print('   - Rainfall: 0-3500mm, Temp: -10-45C, Pest: 0-500k ton')

print('\n3. ENCODING:')
print('   - Area: Target Encoding')
print('   - Item: One-Hot Encoding')

print('\n4. TRAIN-TEST SPLIT:')
print('   - Rekomendasi: Time-Based (1990-2010 train, 2011-2013 test)')

print('\n5. FITUR YEAR:')
print('   - Korelasi lemah (r = -0.02)')
print('   - Rekomendasi: JANGAN gunakan sebagai fitur input')

print('\nFINAL FEATURE SET:')
print('   Input: Area, Item, rainfall, temp, pesticides')
   print('   Target: yield')
   print('   TIDAK MASUKKAN: Year')